# VERITAS Smoke Test - Quick Training Validation

This notebook performs a quick smoke test of the VERITAS implementation by training on a small subset of data.

**Purpose**: Verify that all components work together correctly before full training.

**Test Strategy**:
- Use 100 images from training set (50 authentic, 50 manipulated)
- Train for 2 epochs only
- Verify:
  - Data loading works
  - Model forward pass works
  - Loss computation works
  - Backpropagation works
  - Evaluation metrics work
  - No crashes or errors

**Expected Duration**: 5-10 minutes on GPU

## Setup

In [1]:
import sys
import os
from pathlib import Path

# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/VERITAS/'
    IN_COLAB = True
except ImportError:
    BASE_PATH = './'  # Local testing
    IN_COLAB = False

print(f"Base path: {BASE_PATH}")
print(f"Running in Colab: {IN_COLAB}")

Mounted at /content/drive
Base path: /content/drive/MyDrive/VERITAS/
Running in Colab: True


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import numpy as np
from tqdm import tqdm

import sys
# Add the base path to sys.path so Python can find the 'src' module
sys.path.insert(0, BASE_PATH)

# VERITAS components
from src.config import Config
from src.utils.reproducibility import set_random_seed
from src.utils.environment import verify_environment, print_environment_info
from src.data.annotation_parser import parse_openforensics_json

print("✓ Imports successful")

✓ Imports successful


## Environment Verification

In [3]:
# Verify environment
env_info = verify_environment()
print_environment_info(env_info)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")

ENVIRONMENT INFORMATION

Python version: 3.12.13

Dependency Versions:
  PyTorch: 2.11.0+cu128
  torchvision: 0.26.0+cu128
  OpenCV: 4.13.0
  NumPy: 2.0.2
  scikit-learn: 1.6.1
  scipy: 1.16.3
  Pillow: 11.3.0

CUDA Available: True
GPU Count: 1

GPU 0:
  Model: Tesla T4
  Total VRAM: 14.56 GB

CUDA Version: 12.8
cuDNN Version: 91900


✓ Using device: cuda


In [4]:
# Set random seed for reproducibility
set_random_seed(42)

✓ Random seed set to 42 for reproducibility
  - Python random: 42
  - NumPy: 42
  - PyTorch: 42
  - PyTorch CUDA: 42
  - cuDNN deterministic: True
  - cuDNN benchmark: False


## Load Configuration

In [5]:
# Load config (create if doesn't exist)
config_path = os.path.join(BASE_PATH, 'config.json')

if os.path.exists(config_path):
    config = Config(config_path)
    print(f"✓ Loaded configuration from {config_path}")
else:
    print("⚠ Config not found - run veritas_setup.ipynb first")
    # Create minimal config for smoke test
    from src.config import create_default_config, setup_directories

    setup_directories(BASE_PATH)
    default_config = create_default_config(BASE_PATH)

    import json
    with open(config_path, 'w') as f:
        json.dump(default_config, f, indent=2)

    config = Config(config_path)
    print(f"✓ Created default configuration")

# Override for smoke test
config.set('training.batch_size', 4)  # Small batch for smoke test
config.set('training.num_epochs', 2)   # Just 2 epochs

print(f"\nSmoke test configuration:")
print(f"  Batch size: {config.get('training.batch_size')}")
print(f"  Epochs: {config.get('training.num_epochs')}")
print(f"  Learning rate: {config.get('training.learning_rate')}")

⚠ Config not found - run veritas_setup.ipynb first
✓ Created: /content/drive/MyDrive/VERITAS/dataset
  Purpose: Dataset storage (images and annotations)
✓ Created: /content/drive/MyDrive/VERITAS/checkpoints
  Purpose: Model checkpoint storage
✓ Created: /content/drive/MyDrive/VERITAS/logs
  Purpose: Training logs and metrics
✓ Created: /content/drive/MyDrive/VERITAS/results
  Purpose: Evaluation results and visualizations
✓ Created default configuration
✓ Configuration saved to /content/drive/MyDrive/VERITAS/config.json
✓ Configuration saved to /content/drive/MyDrive/VERITAS/config.json

Smoke test configuration:
  Batch size: 4
  Epochs: 2
  Learning rate: 0.0001


## Create Small Dataset Subset

In [6]:
# Parse annotations
dataset_path = config.get('paths.dataset_path')
train_json = os.path.join(dataset_path, 'Train_poly.json')

print(f"Loading annotations from {train_json}...")
annotations = parse_openforensics_json(train_json)
print(f"✓ Loaded {len(annotations)} annotations")

# Separate by label
authentic = [a for a in annotations if a.label == 0]
manipulated = [a for a in annotations if a.label == 1]

print(f"  Authentic: {len(authentic)}")
print(f"  Manipulated: {len(manipulated)}")

Loading annotations from /content/drive/MyDrive/VERITAS/dataset/Train_poly.json...
✓ Loaded 150866 annotations
  Authentic: 85006
  Manipulated: 65860


In [7]:
# Create small balanced subset
import random

SMOKE_TEST_SIZE = 100  # 50 authentic + 50 manipulated

random.seed(42)
subset_authentic = random.sample(authentic, min(50, len(authentic)))
subset_manipulated = random.sample(manipulated, min(50, len(manipulated)))

smoke_test_annotations = subset_authentic + subset_manipulated
random.shuffle(smoke_test_annotations)

print(f"\n✓ Created smoke test subset: {len(smoke_test_annotations)} images")
print(f"  Authentic: {len(subset_authentic)}")
print(f"  Manipulated: {len(subset_manipulated)}")


✓ Created smoke test subset: 100 images
  Authentic: 50
  Manipulated: 50


## Test Data Loading

In [8]:
# Test loading a few images
from PIL import Image
import matplotlib.pyplot as plt

print("Testing image loading...\n")

test_samples = smoke_test_annotations[:3]

for i, ann in enumerate(test_samples):
    # Note: annotation parser now strips 'Images/' prefix automatically
    img_path = os.path.join(dataset_path, ann.image_path)

    print(f"Sample {i+1}:")
    print(f"  Annotation path: {ann.image_path}")
    print(f"  Full path: {img_path}")

    if os.path.exists(img_path):
        img = Image.open(img_path)
        print(f"  ✓ Image loaded successfully")
        print(f"    Label: {ann.label} ({ann.category})")
        print(f"    Size: {img.size}")
        print(f"    BBox: {ann.bbox}")
        print(f"    Polygon points: {len(ann.polygon) if ann.polygon else 0}")
    else:
        print(f"  ✗ Image not found")
        print(f"    Checked: {img_path}")
    print()

Testing image loading...

Sample 1:
  Annotation path: Train/2389707d44.jpg
  Full path: /content/drive/MyDrive/VERITAS/dataset/Train/2389707d44.jpg
  ✗ Image not found
    Checked: /content/drive/MyDrive/VERITAS/dataset/Train/2389707d44.jpg

Sample 2:
  Annotation path: Train/02037ea01b.jpg
  Full path: /content/drive/MyDrive/VERITAS/dataset/Train/02037ea01b.jpg
  ✓ Image loaded successfully
    Label: 0 (authentic)
    Size: (1024, 710)
    BBox: [290, 217, 56, 64]
    Polygon points: 44

Sample 3:
  Annotation path: Train/1bb2d5a0a6.jpg
  Full path: /content/drive/MyDrive/VERITAS/dataset/Train/1bb2d5a0a6.jpg
  ✓ Image loaded successfully
    Label: 0 (authentic)
    Size: (1024, 768)
    BBox: [476, 266, 53, 58]
    Polygon points: 38



## Test Dataset Class (If Available)

In [9]:
# Try to import and test dataset class
try:
    from src.data.dataset import VERITASDataset

    print("Testing VERITASDataset class...")

    dataset = VERITASDataset(
        annotations=smoke_test_annotations,
        dataset_root=dataset_path,
        config=config,
        split='train'
    )

    print(f"✓ Dataset created with {len(dataset)} samples")

    # Test __getitem__
    print("\nTesting sample loading...")
    sample = dataset[0]

    print(f"✓ Sample loaded:")
    print(f"    Image shape: {sample['image'].shape}")
    print(f"    Mask shape: {sample['mask'].shape}")
    print(f"    Label: {sample['label']}")

    # Test DataLoader
    print("\nTesting DataLoader...")
    dataloader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0
    )

    batch = next(iter(dataloader))
    print(f"✓ Batch loaded:")
    print(f"    Images: {batch['image'].shape}")
    print(f"    Masks: {batch['mask'].shape}")
    print(f"    Labels: {batch['label'].shape}")

    DATASET_WORKS = True

except ImportError:
    print("⚠ VERITASDataset not yet implemented")
    print("  This is OK for early smoke test - will use dummy data")
    DATASET_WORKS = False
except Exception as e:
    print(f"✗ Dataset test failed: {e}")
    DATASET_WORKS = False

Testing VERITASDataset class...
✓ Dataset created with 100 samples

Testing sample loading...
✓ Sample loaded:
    Image shape: torch.Size([3, 600, 600])
    Mask shape: torch.Size([1, 600, 600])
    Label: tensor([1.])

Testing DataLoader...
✓ Batch loaded:
    Images: torch.Size([4, 3, 600, 600])
    Masks: torch.Size([4, 1, 600, 600])
    Labels: torch.Size([4, 1])


## Test Model Architecture (If Available)

In [10]:
# Try to import and test model
try:
    from src.models.veritas import VERITASModel

    print("Testing VERITASModel...")

    model = VERITASModel(config=config)
    model = model.to(device)

    print(f"✓ Model created")

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")

    # Test forward pass with dummy data
    print("\nTesting forward pass...")
    dummy_image = torch.randn(2, 3, 600, 600).to(device)

    with torch.no_grad():
        outputs = model(dummy_image)

    print(f"✓ Forward pass successful:")
    print(f"    Classification output: {outputs['classification_logit'].shape}")
    print(f"    Segmentation output: {outputs['segmentation_logits'].shape}")

    MODEL_WORKS = True

except ImportError:
    print("⚠ VERITASModel not yet implemented")
    print("  This is OK for early smoke test - will use dummy model")
    MODEL_WORKS = False
except Exception as e:
    print(f"✗ Model test failed: {e}")
    import traceback
    traceback.print_exc()
    MODEL_WORKS = False

Testing VERITASModel...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s]


✓ Model created
  Total parameters: 7,254,142
  Trainable parameters: 7,254,142

Testing forward pass...
✓ Forward pass successful:
    Classification output: torch.Size([2, 1])
    Segmentation output: torch.Size([2, 1, 600, 600])


## Test Loss Function (If Available)

In [11]:
# Try to import and test loss
try:
    from src.models.loss import MultiTaskLoss

    print("Testing MultiTaskLoss...")

    loss_fn = MultiTaskLoss(config=config)

    print(f"✓ Loss function created")
    print(f"  Classification weight: {loss_fn.cls_weight}")
    print(f"  Segmentation weight: {loss_fn.seg_weight}")

    # Test loss computation with dummy data
    print("\nTesting loss computation...")
    dummy_cls_logit = torch.randn(2, 1).to(device)
    dummy_seg_logits = torch.randn(2, 1, 600, 600).to(device)
    dummy_labels = torch.randint(0, 2, (2, 1)).float().to(device)
    dummy_masks = torch.randint(0, 2, (2, 1, 600, 600)).float().to(device)

    predictions = {
        'classification_logit': dummy_cls_logit,
        'segmentation_logits': dummy_seg_logits
    }

    targets = {
        'label': dummy_labels,
        'mask': dummy_masks
    }

    losses = loss_fn(predictions, targets)

    print(f"✓ Loss computation successful:")
    print(f"    Total loss: {losses['total_loss'].item():.4f}")
    print(f"    Classification loss: {losses['classification_loss'].item():.4f}")
    print(f"    Segmentation loss: {losses['segmentation_loss'].item():.4f}")

    LOSS_WORKS = True

except ImportError:
    print("⚠ MultiTaskLoss not yet implemented")
    print("  This is OK for early smoke test - will use simple BCE loss")
    LOSS_WORKS = False
except Exception as e:
    print(f"✗ Loss test failed: {e}")
    import traceback
    traceback.print_exc()
    LOSS_WORKS = False

Testing MultiTaskLoss...
✓ Loss function created
  Classification weight: 0.4
  Segmentation weight: 0.6

Testing loss computation...
✓ Loss computation successful:
    Total loss: 0.7483
    Classification loss: 0.8933
    Segmentation loss: 0.6517


## Smoke Test Training Loop

Run a minimal training loop to verify all components work together.

In [12]:
if DATASET_WORKS and MODEL_WORKS and LOSS_WORKS:
    print("=" * 60)
    print("RUNNING SMOKE TEST TRAINING")
    print("=" * 60)

    # Setup
    model.train()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.get('training.learning_rate'),
        weight_decay=config.get('training.weight_decay')
    )

    num_epochs = config.get('training.num_epochs')

    # Training loop
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")

        epoch_losses = []

        pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}")
        for batch_idx, batch in enumerate(pbar):
            # Move to device
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)
            labels = batch['label'].to(device)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(images)

            # Compute loss
            targets = {'label': labels, 'mask': masks}
            losses = loss_fn(outputs, targets)

            # Backward pass
            losses['total_loss'].backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                config.get('training.gradient_clip_max_norm')
            )

            optimizer.step()

            # Track loss
            epoch_losses.append(losses['total_loss'].item())

            # Update progress bar
            pbar.set_postfix({
                'loss': losses['total_loss'].item(),
                'cls_loss': losses['classification_loss'].item(),
                'seg_loss': losses['segmentation_loss'].item()
            })

        # Epoch summary
        avg_loss = np.mean(epoch_losses)
        print(f"  Average loss: {avg_loss:.4f}")

    print("\n" + "=" * 60)
    print("✓ SMOKE TEST TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print("\nAll components working correctly:")
    print("  ✓ Data loading")
    print("  ✓ Model forward pass")
    print("  ✓ Loss computation")
    print("  ✓ Backpropagation")
    print("  ✓ Optimizer step")
    print("\nReady for full training!")

else:
    print("\n" + "=" * 60)
    print("SMOKE TEST SKIPPED")
    print("=" * 60)
    print("\nSome components not yet implemented:")
    print(f"  Dataset: {'✓' if DATASET_WORKS else '✗'}")
    print(f"  Model: {'✓' if MODEL_WORKS else '✗'}")
    print(f"  Loss: {'✓' if LOSS_WORKS else '✗'}")
    print("\nImplement missing components and re-run this notebook.")

RUNNING SMOKE TEST TRAINING

Epoch 1/2


Epoch 1: 100%|██████████| 25/25 [01:15<00:00,  3.02s/it, loss=0.652, cls_loss=0.568, seg_loss=0.708]


  Average loss: 0.7183

Epoch 2/2


Epoch 2: 100%|██████████| 25/25 [00:06<00:00,  3.74it/s, loss=0.717, cls_loss=0.787, seg_loss=0.67]

  Average loss: 0.6363

✓ SMOKE TEST TRAINING COMPLETED SUCCESSFULLY

All components working correctly:
  ✓ Data loading
  ✓ Model forward pass
  ✓ Loss computation
  ✓ Backpropagation
  ✓ Optimizer step

Ready for full training!


## Test Model Inference

In [13]:
if MODEL_WORKS:
    print("Testing inference mode...\n")

    model.eval()

    with torch.no_grad():
        # Get a batch
        if DATASET_WORKS:
            batch = next(iter(dataloader))
            images = batch['image'][:2].to(device)  # Take 2 samples
        else:
            images = torch.randn(2, 3, 600, 600).to(device)

        # Run inference
        outputs = model(images)

        # Get predictions
        cls_probs = torch.sigmoid(outputs['classification_logit'])
        seg_probs = torch.sigmoid(outputs['segmentation_logits'])

        # Binary decisions
        cls_preds = (cls_probs >= 0.5).long()
        seg_masks = (seg_probs >= 0.5).long()

        print("✓ Inference successful")
        print(f"\nSample predictions:")
        for i in range(min(2, cls_probs.shape[0])):
            print(f"  Image {i + 1}:")
            print(f"    Classification: {cls_preds[i].item()} (prob: {cls_probs[i].item():.3f})")
            print(f"    Segmentation: {seg_masks[i].sum().item()} manipulated pixels")
else:
    print("⚠ Skipping inference test - model not available")

Testing inference mode...

✓ Inference successful

Sample predictions:
  Image 1:
    Classification: 1 (prob: 0.556)
    Segmentation: 37230 manipulated pixels
  Image 2:
    Classification: 0 (prob: 0.443)
    Segmentation: 29931 manipulated pixels


## Summary

In [14]:
print("=" * 60)
print("SMOKE TEST SUMMARY")
print("=" * 60)
print("\nComponent Status:")
print(f"  Environment setup: ✓")
print(f"  Configuration: ✓")
print(f"  Annotation parsing: ✓")
print(f"  Dataset class: {'✓' if DATASET_WORKS else '✗ Not implemented'}")
print(f"  Model architecture: {'✓' if MODEL_WORKS else '✗ Not implemented'}")
print(f"  Loss function: {'✓' if LOSS_WORKS else '✗ Not implemented'}")

if DATASET_WORKS and MODEL_WORKS and LOSS_WORKS:
    print("\n✓ ALL SYSTEMS GO - Ready for full training!")
else:
    print("\n⚠ Some components need implementation before full training")
    print("\nNext steps:")
    if not DATASET_WORKS:
        print("  1. Implement VERITASDataset (Task 8)")
    if not MODEL_WORKS:
        print("  2. Implement VERITASModel (Tasks 10-16)")
    if not LOSS_WORKS:
        print("  3. Implement MultiTaskLoss (Task 18)")

print("\n" + "=" * 60)

SMOKE TEST SUMMARY

Component Status:
  Environment setup: ✓
  Configuration: ✓
  Annotation parsing: ✓
  Dataset class: ✓
  Model architecture: ✓
  Loss function: ✓

✓ ALL SYSTEMS GO - Ready for full training!

